In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
trips = pd.read_csv('s3://personal-ds-data-sandbox/trippub.csv', low_memory = False)
persons = pd.read_csv('s3://personal-ds-data-sandbox/perpub.csv', low_memory = False)
hh = pd.read_csv('s3://personal-ds-data-sandbox/hhpub.csv', low_memory = False)

In [ ]:
print(trips.shape)
print(persons.shape)
print(hh.shape)

In [ ]:
trips.head()

In [ ]:
trips.columns

In [ ]:
print([col for col in trips.columns if 'WHY' in col])

In [ ]:
persons.head()

In [ ]:
missing_codes = [-1, -7, -8, -9, -88, -99]
for df in [trips, persons, hh]:
    df.replace(missing_codes, np.nan, inplace=True)

In [ ]:
hh.head()

In [ ]:
hh_cols = ['HOUSEID', 'HHFAMINC', 'HHVEHCNT', 'HHSIZE', 'URBAN', 'URBANSIZE', 'HOMEOWN', 'TRAVDAY', 'HBRESDN', 'HBHUR','HTHTNRNT','HTPPOPDN','HTRESDN','HTEEMPDN','HBHTNRNT', 'HBPPOPDN','HBRESDN']

In [ ]:
hh_filt = hh[hh_cols].copy()

In [ ]:
hh_filt.head()

In [ ]:
per_cols = ['HOUSEID', 'PERSONID', 'R_SEX', 'R_AGE', 'PRMACT', 'WORKER', 'EDUC','PAYPROF', 'DRIVER']
per_filt = persons[per_cols].copy()
per_filt['SEX'] = per_filt['R_SEX'].map({1:0, 2:1})

In [ ]:
per_filt.drop('R_SEX', axis=1)

In [ ]:
trip_cols = ['HOUSEID', 'PERSONID', 'TRPTRANS', 'TRVLCMIN', 'TRPMILES', 
    'STRTTIME', 'WHYTRP1S', 'DWELTIME', 
    'OBRESDN', 'DBRESDN', 'DTEEMPDN', 'ENDTIME']
trips_filt = trips[trip_cols].copy()

In [ ]:
df_m1 = pd.merge(trips_filt, per_filt, on=['HOUSEID','PERSONID'], how='left')

In [ ]:
df = pd.merge(df_m1, hh_filt, on = 'HOUSEID', how='left')

In [ ]:
df.head()

In [ ]:
print(df.isnull().sum())

In [ ]:
# Simple binary — recommended
df["has_license"] = (df["DRIVER"] == 1).astype(int)
# 1 = has license, 0 = no license

In [ ]:
# Drop the columns that are mostly empty
cols_to_delete = ['PAYPROF', 'DWELTIME', 'PRMACT', 'WORKER', 'EDUC', 'HBRESDN']
df = df.drop(columns=cols_to_delete)

# Drop rows where essential trip info is missing
df = df.dropna(subset=['TRPTRANS', 'TRVLCMIN', 'TRPMILES', 'R_SEX', 'R_AGE', 'HBHTNRNT'])

density_cols = ['OBRESDN', 'DBRESDN', 'DTEEMPDN', 'HBPPOPDN', 'HTEEMPDN', 'HHFAMINC', 'HTRESDN',]

for col in density_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
# Function to convert HHMM to total minutes
def to_mins(hhmm):
    return (hhmm // 100) * 60 + (hhmm % 100)

# Create new continuous features
df['start_mins'] = df['STRTTIME'].apply(to_mins)
df['end_mins'] = df['ENDTIME'].apply(to_mins)

# Calculate duration to verify against TRVLCMIN (Sanity Check)
df['duration'] = df['end_mins'] - df['start_mins']

# Create your Departure Hour feature for the model
df['dep_hour'] = df['STRTTIME'] // 100

In [ ]:
def estimate_cost(row):
    # If they took a car (Codes 03-06)
    if row['TRPTRANS'] in [3, 4, 5, 6]:
        cost = row['TRPMILES'] * 0.18 # Estimated parking fee
        return cost
    # If they took transit (Codes 10-16)
    elif 10 <= row['TRPTRANS'] <= 16:
        return 2.50 # Standard fare estimate
    return 0

df['trip_cost'] = df.apply(estimate_cost, axis=1)

In [ ]:
def estimate_wait_time(row):
    if row['HBHUR'] == 'U':
        return 7  # Average urban wait
    elif row['HBHUR'] == 'S':
        return 20 # Average suburban wait
    else:
        return 45 # Rural/Town wait
    
df['Est_Wait_Time'] = df.apply(estimate_wait_time, axis=1)

In [ ]:
def get_period(hr):
    if 6 <= hr <= 9: return 'AM_Peak'
    if 15 <= hr <= 19: return 'PM_Peak'
    if 9 < hr < 15: return 'Mid_Day'
    return 'Night'

df['time_period'] = (df['STRTTIME'] // 100).apply(get_period)

In [ ]:
# Calculate Miles Per Hour (MPH)
df['trip_speed_mph'] = df['TRPMILES'] / (df['TRVLCMIN'] / 60)

# Handle cases where TRVLCMIN is 0 to avoid division errors
df['trip_speed_mph'] = df['trip_speed_mph'].replace([np.inf, -np.inf], 0)

In [ ]:
# Fix 1 — HHFAMINC income bracket to midpoint
income_map = {
    1: 5000,   2: 12500,  3: 20000,  4: 30000,   5: 42500,
    6: 62500,  7: 87500,  8: 112500, 9: 137500, 10: 175000, 11: 225000
}
df["income"] = df["HHFAMINC"].map(income_map)

In [ ]:
# Check the counts for each purpose code
print(df['WHYTRP1S'].value_counts().sort_index())

In [ ]:
print(df['TRPTRANS'].value_counts().sort_index())

In [ ]:
# Create a lean mapping
summary_purpose_map = {
    1: 'Work',
    10: 'Home',
    20: 'Shop',
    30: 'Personal',
    40: 'School',
    50: 'Medical',
    70: 'Social',
    80: 'Other',
    97: 'Other'
}

# Apply the map
df['TRPPRP'] = df['WHYTRP1S'].map(summary_purpose_map)

In [ ]:
# Assuming you have already mapped WHYTRP1S to 'purp_label'
# and R_SEX to 'gender_label'

cols_to_encode = ['TRPPRP', 'HBHUR', 'time_period']

# This command creates the 1s and 0s and DROPS the original text columns
df_enc = pd.get_dummies(df, columns=cols_to_encode, prefix=['purp', 'urb', 'time'])

# Check your new columns
print(df_enc.columns)

In [ ]:
df_enc.head()

In [ ]:
df_enc.columns

In [ ]:
# Fill Density with the Median
df_enc['HBPPOPDN'] = df_enc['HBPPOPDN'].fillna(df_enc['HBPPOPDN'].median())

# Fill Vehicle Count with the most common number (Mode)
df_enc['HHVEHCNT'] = df_enc['HHVEHCNT'].fillna(df_enc['HHVEHCNT'].mode()[0])

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Identify only the "Large" numerical columns
# (Add or remove from this list as needed based on your 40+ columns)
cols_to_scale = [
    'TRVLCMIN', 'TRPMILES', 'trip_cost', 
    'OBRESDN', 'DBRESDN', 'HTEEMPDN', 'DTEEMPDN', 'HHVEHCNT', 'HHSIZE','HTHTNRNT','HTPPOPDN','HTRESDN','HBHTNRNT', 'HBPPOPDN', 'start_mins', 'trip_speed_mph','income'
]

# 2. Initialize the Scaler
scaler = StandardScaler()

# 3. Apply scaling to those specific columns
# This replaces the large numbers with values typically between -3 and 3
df_enc[cols_to_scale] = scaler.fit_transform(df_enc[cols_to_scale])

In [ ]:
df_enc.head()

In [ ]:
df_enc.columns

In [ ]:
drop = ['HOUSEID', 'PERSONID','STRTTIME','ENDTIME', 'R_SEX', 'WHYTRP1S', 'HHFAMINC','DRIVER']
df_final = df_enc.drop(columns=drop)

In [ ]:
mode_map = {
    # Active
    1: "Active",    # walk
    2: "Active",    # bike

    # Auto
    3: "Auto",      # car driver
    4: "Auto",      # car passenger
    5: "Auto",      # van
    6: "Auto",      # pickup truck
    20: "Auto",     # taxi/Uber ← merge here if you want to keep it

    # Transit (public only)
    11: "Transit",  # public bus
    12: "Transit",  # paratransit
    14: "Transit",  # city to city bus
    15: "Transit",  # Amtrak
    16: "Transit",  # commuter rail
    17: "Transit",  # subway / metro
    18: "Transit",  # elevated rail
    19: "Transit",  # streetcar / trolley

    # Dropped entirely:
    # 7  → golf cart
    # 8  → motorcycle
    # 9  → RV
    # 10 → school bus (captive, not a choice)
    # 13 → charter/private bus
}


In [ ]:
# 2. Apply the Map
df_final['mode_label'] = df_final['TRPTRANS'].map(mode_map)

# 3. Handle the "Other" and -9 values
df_final = df_final.dropna(subset=['mode_label'])
print(df_final["mode_label"].value_counts())

In [ ]:
# 4. Final Encoding for the Neural Network
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_final['target'] = le.fit_transform(df_final['mode_label'])

# Check your classes: 0 might be Active, 1 might be Auto, etc.
print(dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
df_final.to_csv("Data.csv", index = False)